# Paper 1 UIE — Frozen V7 Image Demo

Run the frozen V7 model on your own underwater image. The EUVP dataset and a reference image are not needed. A GPU is recommended; CPU also works.

In [ ]:
!git clone https://github.com/HariniS1207/Paper1_UIE.git
%cd Paper1_UIE

In [ ]:
!pip install -r requirements.txt

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Upload an underwater image

Select one JPG or PNG image. The demo preprocessing converts it to RGB and resizes it to 256 × 256 using Lanczos, matching the V7 web demo.

In [ ]:
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise ValueError('Upload one underwater JPG or PNG image.')
image_name = next(iter(uploaded))
print('Using:', image_name)

In [ ]:
import hashlib
import numpy as np
import torch
from PIL import Image, ImageOps
from IPython.display import display
from pathlib import Path
import sys

root = Path('/content/Paper1_UIE')
sys.path.insert(0, str(root / 'src'))
from models.paper1_model_v7 import Paper1ModelV7

checkpoint_path = root / 'checkpoints/v7/paper1_v7_best_epoch_019.pth'
expected_sha256 = 'A0E04FEEC9A6C69CCEE0DD8A6E23E9C35227CDAA10C270DD82F853536920B49E'
actual_sha256 = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest().upper()
if actual_sha256 != expected_sha256:
    raise RuntimeError(f'Checkpoint SHA-256 mismatch: {actual_sha256}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
if int(checkpoint.get('epoch', -1)) != 19:
    raise RuntimeError(f"Expected epoch 19 checkpoint, got {checkpoint.get('epoch')}")
model = Paper1ModelV7().to(device)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval()

with Image.open(image_name) as source:
    original = ImageOps.exif_transpose(source).convert('RGB')
    prepared = original.resize((256, 256), Image.Resampling.LANCZOS)
array = np.asarray(prepared, dtype=np.float32) / 255.0
x = torch.from_numpy(array.transpose(2, 0, 1)).unsqueeze(0).to(device)
with torch.inference_mode():
    output = model(x)

def to_image(tensor):
    image_array = tensor.detach().float().cpu().clamp(0, 1)[0].permute(1, 2, 0).numpy()
    return Image.fromarray(np.round(image_array * 255).astype(np.uint8))

names = ('Conservative', 'Balanced', 'Aggressive')
selected_index = int(output['selected_index'][0].item())
probabilities = output['selection_weights'][0].detach().cpu().numpy()
print('Selected candidate:', names[selected_index])
print('Selection probabilities:', {name: f'{prob:.2%}' for name, prob in zip(names, probabilities)})

print('Input (demo-preprocessed)')
display(prepared)
for name in names:
    print(name)
    display(to_image(output['candidates'][name.lower()]))
print('Selected output:', names[selected_index])
display(to_image(output['selected_image']))

The notebook verifies the frozen checkpoint SHA-256 before loading it. Save any output you want to keep from the displayed images or rerun inference locally with `demo/app_v7.py`. For limitations and the held-out results, see `docs/V7_MODEL_CARD.md` and `results/v7/v7_final_test_report.md`.